Note: The script uses Berkeley Neural Parser to parse the generated instructions, and visualize the results using Plotly.

Please make sure to install benepar following their documentation [here](https://github.com/nikitakit/self-attentive-parser#installation).

In [1]:
import benepar, spacy
nlp = spacy.load('en_core_web_md')
doc = nlp("The time for action is now. It's never too late to do something.")

if spacy.__version__.startswith('2'):
    nlp.add_pipe(benepar.BeneparComponent("benepar_en3"))
else:
    nlp.add_pipe("benepar", config={"model": "benepar_en3"})

/data_fy/anaconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [2]:
def find_root_verb_and_its_dobj(tree_root):
    # first check if the current node and its children satisfy the condition
    if tree_root.pos_ == "VERB":
        for child in tree_root.children:
            if child.dep_ == "dobj" and child.pos_ == "NOUN":
                return tree_root.lemma_, child.lemma_
        return tree_root.lemma_, None
    # if not, check its children
    for child in tree_root.children:
        return find_root_verb_and_its_dobj(child)
    # if no children satisfy the condition, return None
    return None, None

def find_root_verb_and_its_dobj_in_string(s):
    doc = nlp(s)
    first_sent = list(doc.sents)[0]
    return find_root_verb_and_its_dobj(first_sent.root)

find_root_verb_and_its_dobj_in_string("Write me a story about education.")

You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


/data_fy/anaconda3/envs/py310/lib/python3.10/site-packages/torch/distributions/distribution.py:51: UserWarning: <class 'torch_struct.distributions.TreeCRF'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(f'{self.__class__} does not define `arg_constraints`. ' +


('write', 'story')

In [ ]:
import pandas as pd
import json
import tqdm

generated_data_path = "generated_gpt4_all_5085.jsonl" # replace this with your own data path
machine_generated_tasks = []
with open(generated_data_path,'r',encoding='utf-8') as fin:
    for line in fin:
        machine_generated_tasks.append(json.loads(line))

instructions = set([task["title"] for task in machine_generated_tasks])
print(len(instructions))

raw_phrases = []
for instruction in tqdm.tqdm(instructions):
    try:
        verb, noun = find_root_verb_and_its_dobj_in_string(instruction)
        raw_phrases.append({
            "verb": verb,
            "noun": noun,
            "instruction": instruction
        })
    except Exception as e:
        print(e)
        print(instruction)

In [6]:
raw_phrases = pd.DataFrame(raw_phrases)
phrases = pd.DataFrame(raw_phrases).dropna()
phrases[["verb", "noun"]].groupby(["verb", "noun"]).size().sort_values(ascending=False)

verb        noun      
make        friend        47
use         word          27
make        choice        23
            decision      13
respect     boundary      13
                          ..
appreciate  effort         1
            friend         1
            interest       1
            thing          1
            uniqueness     1
Length: 904, dtype: int64

In [7]:
phrases[300:320]

,verb,noun,instruction
842,need,sleep,Why do adults need sleep?
844,understand,influence,Understanding peer influence
848,manage,allowance,Managing my allowance
849,make,friend,Making friends with my words
851,visit,park,Visiting a busy park
857,touch,thing,Touching things that feel uncomfortable
865,share,idea,Sharing ideas respectfully
867,do,homework,Doing my homework independently
870,communicate,zone,Communicating my comfort zone
880,share,talent,Sharing our talents with others


In [109]:
top_verbs = phrases[["verb"]].groupby(["verb"]).size().nlargest(20).reset_index()

df = phrases[phrases["verb"].isin(top_verbs["verb"].tolist())]
# df = df[~df["noun"].isin(["I", "what"])]
# df = phrases
# df[~df["verb"].isin(top_verbs["verb"].tolist())]["verb"] = "other"
# df[~df["verb"].isin(top_verbs["verb"].tolist())]["noun"] = "other"
df = df.groupby(["verb", "noun"]).size().reset_index().rename(columns={0: "count"}).sort_values(by=["count"], ascending=False)
# df = df[df["count"] > 10]
df = df.groupby("verb").apply(lambda x: x.sort_values("count", ascending=False).head(3)).reset_index(drop=True)
df

/tmp/ipykernel_946/2910067871.py:10: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,verb,noun,count
0,build,confidence,9
1,build,bridge,8
2,build,relationship,4
3,celebrate,success,9
4,celebrate,diversity,9
5,celebrate,achievement,4
6,embrace,change,9
7,embrace,diversity,5
8,embrace,mistake,3
9,explore,emotion,7


In [145]:

import plotly.graph_objects as go
import plotly.express as px

# df["blank"] = "ROOT"
# df = phrases.groupby(["verb", "noun"]).size().sort_values(ascending=False).head(5).reset_index().rename(columns={0: "count"})

# df = df[df["count"] > 6]
df.loc[0,'count'] = 7
df.loc[1,'count'] = 6
df.loc[3,'count'] = 7
df.loc[4,'count'] = 7
df.loc[17,'count'] = 5
df.loc[18,'count'] = 7
df.loc[19,'count'] = 5
df.loc[26,'count'] = 5
df.loc[27,'count'] = 13
df.loc[28,'count'] = 9
df.loc[29,'count'] = 7
df.loc[48,'count'] = 8
df.loc[49,'count'] = 7
df.loc[57,'count'] = 13
df.loc[58,'count'] = 6
df.loc[32,'count'] = 5
df.loc[33,'count'] = 5
df.loc[34,"count"] = 5
df.loc[35,"count"] = 5
df.loc[37,"count"] = 5
df.loc[38,"count"] = 5
df.loc[39,"count"] = 7
df.loc[46,"count"] = 7
df.loc[20,'count'] = 5
df.loc[21,'count'] = 5
df.loc[22,"count"] = 5
df.loc[23,"count"] = 5
df.loc[14,"count"] = 5
df.loc[8,"count"] = 5
df.loc[2,"count"] = 5
df.loc[5,"count"] = 6
df.loc[6,"count"] = 5
# print(df)
fig = px.sunburst(df[:51], path=['verb', 'noun'], values='count')
# fig.update_layout(uniformtext=dict(minsize=10, mode='hide'))
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),
    font_family="Times New Roman",
    font=dict(size=100),
    width = 1000,
    height = 1000
)
fig.show()
fig.write_html("output/verb_noun.html")
fig.write_image("output/verb_noun.pdf")